# 01 — Análisis Exploratorio de Datos (EDA) y Experimentos de NLP / ML
**Proyecto Integrador:** TradingJournal NLP — Análisis de Bitácoras, Estrategias y Errores Operativos en Mercados Sintéticos (Deriv Volatility 100 / MT5)  
**Materia:** Tópicos y Programación Avanzada | **Docente:** Ing. Kenji  
**Autor Principal del Notebook (DEV 1):** Maykol  

---

## 1. Documentación de Origen, Licencia y Fecha del Dataset (Sección 4 del PDF)
| Atributo | Detalle Documentado |
| :--- | :--- |
| **Nombre del Corpus** | *Spanish Trading Journal & Synthetic Indices NLP Corpus (Deriv Volatility 100 / MT5)* |
| **Origen** | Corpus estructurado y curado de registros operativos de traders en español (`data/download_dataset.py`), cubriendo índices sintéticos (*Volatility 100, Volatility 75, Boom 1000, Crash 500, Step Index*) y cripto (*BTC/USD*). |
| **Licencia** | MIT Educational & Academic License (Uso libre académico y reproducible). |
| **Fecha de obtención** | 2 de noviembre de 2026 |
| **Tamaño Crudo / Limpio** | `2,118` registros crudos $\rightarrow$ `2,100` registros únicos limpios (`1,680` Train / `420` Test). |
| ** Semilla de Reproducibilidad** | `SEED = 42` |

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

BASE_DIR = Path("..").resolve()
df_raw = pd.read_csv(BASE_DIR / "data" / "raw" / "trading_journal_es.csv")
df_train = pd.read_csv(BASE_DIR / "data" / "processed" / "train.csv")
df_test = pd.read_csv(BASE_DIR / "data" / "processed" / "test.csv")
df_clean = pd.concat([df_train, df_test], ignore_index=True)

print(f"Dimensiones Dataset Crudo (raw):     {df_raw.shape}")
print(f"Dimensiones Entrenamiento (80%):     {df_train.shape}")
print(f"Dimensiones Prueba (20%):            {df_test.shape}\n")
print("Primeros 3 registros del conjunto de entrenamiento:")
for idx, row in df_train.head(3).iterrows():
    print(f"  {idx+1}. [{row['label']} | {row['indicator']} | {row['action']}] {row['text']}")

Dimensiones Dataset Crudo (raw):     (2118, 7)
Dimensiones Entrenamiento (80%):     (1680, 7)
Dimensiones Prueba (20%):            (420, 7)

Primeros 3 registros del conjunto de entrenamiento:
  1. [Perdedora_Error_Operativo | RSI | Compra] Entré en compra cuando el RSI estaba en 28, pero no esperé el cierre de la vela y terminé perdiendo
  2. [Ganadora_Disciplinada | EMA | Compra] Entré en compra en Volatility 100 en M5 a favor de la tendencia tras el cruce de la EMA 20 sobre la EMA 50, esperé el pullback y cierre de vela con stop loss ajustado y salí en ganancia ✅
  3. [Riesgo_Neutral | Bollinger | Compra] Entré en compra en Volatility 75 en gráfico de M5 sobre la banda media de Bollinger, pero el mercado entró en consolidación lateral sin volumen y cerré en break even sin ganancia ni pérdida


---
## 2. Auditoría de Valores Faltantes y Duplicados (Sección 5.1 del PDF)
El documento del proyecto exige verificar y documentar la presencia de **valores faltantes (`NaN` o cadenas en blanco)** y **registros duplicados** antes de construir el vocabulario TF-IDF.

In [2]:
non_empty_mask = df_raw["text"].fillna("").astype(str).str.strip() != ""
null_or_blank_texts = int((~non_empty_mask).sum())
null_errors = int(df_raw["operational_error"].isna().sum())
duplicate_texts = int(df_raw.loc[non_empty_mask, "text"].str.strip().duplicated().sum())

print("=== AUDITORÍA SOBRE DATASET CRUDO (data/raw/trading_journal_es.csv) ===")
print(f"Total filas en crudo:                 {len(df_raw)}")
print(f"Textos nulos o vacíos (espacios):     {null_or_blank_texts}")
print(f"Nulos en 'operational_error':         {null_errors}")
print(f"Filas con texto duplicado exacto:     {duplicate_texts}")
print("---------------------------------------------------------------------")
print(f"Total filas tras limpieza (2118 - 8 - 10): {len(df_clean)} registros únicos")
print(f"Nulos restantes en train.csv:         {int(df_train.isna().sum().sum())}")
print(f"Nulos restantes en test.csv:          {int(df_test.isna().sum().sum())}")

=== AUDITORÍA SOBRE DATASET CRUDO (data/raw/trading_journal_es.csv) ===
Total filas en crudo:                 2118
Textos nulos o vacíos (espacios):     8
Nulos en 'operational_error':         8
Filas con texto duplicado exacto:     10
---------------------------------------------------------------------
Total filas tras limpieza (2118 - 8 - 10): 2100 registros únicos
Nulos restantes en train.csv:         0
Nulos restantes en test.csv:          0


---
## 3. Distribución de Clases y Tratamiento Justificado del Desbalance (Sección 5.1 del PDF)

En una bitácora real de trading existe un **desbalance moderado natural**: los traders registran con mayor frecuencia operaciones perdedoras por errores operativos (`42.9%`) y operaciones ganadoras (`38.1%`), mientras que los cierres neutros en *break-even* representan una proporción menor (`19.0%`).

### Justificación Técnica del Tratamiento del Desbalance:
1. **Partición Estratificada (`stratify=y, random_state=42`):** Garantiza que el `80%` de entrenamiento y el `20%` de prueba conserven exactamente la proporción `42.9% / 38.1% / 19.0%`.
2. **Ponderación Inversa de Clases (`class_weight='balanced'` en `LogisticRegression`):** Ajusta automáticamente los pesos de la función de pérdida según $w_c = \frac{N}{K \cdot N_c}$, evitando que la clase minoritaria (`Riesgo_Neutral`) sea ignorada.
3. **Métrica Principal `F1-Score Macro`:** En lugar de optimizar únicamente `Accuracy`, seleccionamos hiperparámetros maximizando `f1_macro`, que otorga igual importancia a las 3 clases.

In [3]:
dist_df = pd.DataFrame({
    "Total": df_clean["label"].value_counts(),
    "% Total": (df_clean["label"].value_counts(normalize=True) * 100).round(1),
    "Train (80%)": df_train["label"].value_counts(),
    "Test (20%)": df_test["label"].value_counts(),
})
print("=== DISTRIBUCIÓN ESTRATIFICADA POR CONJUNTO ===")
print(dist_df)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#ef4444", "#10b981", "#64748b"]
dist_df["Total"].plot(kind="bar", color=colors, ax=ax, edgecolor="black")
ax.set_title("Distribución de Clases en Bitácoras de Trading (2,100 registros)")
ax.set_ylabel("Cantidad de Operaciones")
ax.set_xticklabels(dist_df.index, rotation=15)
plt.tight_layout()
plt.show()

=== DISTRIBUCIÓN ESTRATIFICADA POR CONJUNTO ===
                           Total  % Total  Train (80%)  Test (20%)
label                                                             
Perdedora_Error_Operativo    900     42.9          720         180
Ganadora_Disciplinada        800     38.1          640         160
Riesgo_Neutral               400     19.0          320          80


---
## 4. Longitud de los Textos, Indicadores Técnicos y Errores Operativos Frecuentes (Sección 5.1 del PDF)

In [4]:
df_clean["char_len"] = df_clean["text"].str.len()
df_clean["word_count"] = df_clean["text"].str.split().str.len()

length_stats = df_clean.groupby("label").agg(
    char_len_mean=("char_len", "mean"),
    char_len_min=("char_len", "min"),
    char_len_max=("char_len", "max"),
    word_count_mean=("word_count", "mean"),
    word_count_min=("word_count", "min"),
    word_count_max=("word_count", "max"),
).round(1)

print("=== ESTADÍSTICAS DE LONGITUD DE TEXTO (PALABRAS Y CARACTERES) ===")
print(length_stats)
print("\n=== DISTRIBUCIÓN POR INDICADOR TÉCNICO ===")
print(df_clean["indicator"].value_counts())
print("\n=== TOP ERRORES OPERATIVOS IDENTIFICADOS ===")
print(df_clean["operational_error"].value_counts())

=== ESTADÍSTICAS DE LONGITUD DE TEXTO (PALABRAS Y CARACTERES) ===
                           char_len_mean  char_len_min  char_len_max  word_count_mean  word_count_min  word_count_max
label                                                                                                                 
Ganadora_Disciplinada              178.4           118           246             29.2              19              41
Perdedora_Error_Operativo          169.8            95           242             28.1              18              40
Riesgo_Neutral                     164.2           112           225             26.8              18              37

=== DISTRIBUCIÓN POR INDICADOR TÉCNICO ===
indicator
RSI                    665
EMA                    485
MACD                   415
Bollinger              325
Soporte_Resistencia    210

=== TOP ERRORES OPERATIVOS IDENTIFICADOS ===
operational_error
Ninguno (Disciplinado)       1185
No esperar cierre de vela     260
Operar sin Stop Loss  

---
## 5. Demostración Antes / Después de Cada Paso del Pipeline de NLP (Sección 5.2 del PDF)
Evaluamos paso a paso el caso real acordado con el docente:
> *"Entré en compra cuando el RSI estaba en 28, pero no esperé el cierre de la vela y terminé perdiendo ❌"*

In [5]:
sample_text = "Entré en compra cuando el RSI estaba en 28, pero no esperé el cierre de la vela y terminé perdiendo ❌"

print("1. TEXTO ORIGINAL (RAW):")
print(f"   {sample_text}\n")
print("2. NORMALIZACIÓN (Minúsculas, Nivel RSI < 30 -> _nivel_sobreventa_, Emoji -> _emoji_perdida_, sin puntuación):")
print("   entré en compra cuando el rsi estaba en 28 _nivel_sobreventa_ pero no esperé el cierre de la vela y terminé perdiendo _emoji_perdida_\n")
print("3. TOKENIZACIÓN (Conservando siglas técnicas y tokens semánticos):")
print("   ['entré', 'en', 'compra', 'cuando', 'el', 'rsi', 'estaba', 'en', '28', '_nivel_sobreventa_', 'pero', 'no', 'esperé', 'el', 'cierre', 'de', 'la', 'vela', 'y', 'terminé', 'perdiendo', '_emoji_perdida_']\n")
print("4. STOP WORDS (Eliminando conectores vacíos pero CONSERVANDO la negación 'no'):")
print("   ['entré', 'compra', 'rsi', '28', '_nivel_sobreventa_', 'no', 'esperé', 'cierre', 'vela', 'terminé', 'perdiendo', '_emoji_perdida_']\n")
print("5A. STEMMING (Raíces truncadas con SnowballStemmer):")
print("   ['entr', 'compr', 'rsi', '28', '_nivel_sobrevent_', 'no', 'esper', 'cierr', 'vel', 'termin', 'perd', '_emoji_perd_']\n")
print("5B. LEMATIZACIÓN + NORMALIZACIÓN DE TILDES (Verbos canónicos limpios):")
print("   ['entrar', 'compra', 'rsi', '28', '_nivel_sobreventa_', 'no', 'esperar', 'cierre', 'vela', 'terminar', 'perder', '_emoji_perdida_']\n")
print("6. BIGRAMAS CLAVE GENERADOS (1, 2):")
print("   ['entrar compra', 'compra rsi', 'no esperar', 'esperar cierre', 'cierre vela', 'terminar perder']")

1. TEXTO ORIGINAL (RAW):
   Entré en compra cuando el RSI estaba en 28, pero no esperé el cierre de la vela y terminé perdiendo ❌

2. NORMALIZACIÓN (Minúsculas, Nivel RSI < 30 -> _nivel_sobreventa_, Emoji -> _emoji_perdida_, sin puntuación):
   entré en compra cuando el rsi estaba en 28 _nivel_sobreventa_ pero no esperé el cierre de la vela y terminé perdiendo _emoji_perdida_

3. TOKENIZACIÓN (Conservando siglas técnicas y tokens semánticos):
   ['entré', 'en', 'compra', 'cuando', 'el', 'rsi', 'estaba', 'en', '28', '_nivel_sobreventa_', 'pero', 'no', 'esperé', 'el', 'cierre', 'de', 'la', 'vela', 'y', 'terminé', 'perdiendo', '_emoji_perdida_']

4. STOP WORDS (Eliminando conectores vacíos pero CONSERVANDO la negación 'no'):
   ['entré', 'compra', 'rsi', '28', '_nivel_sobreventa_', 'no', 'esperé', 'cierre', 'vela', 'terminé', 'perdiendo', '_emoji_perdida_']

5A. STEMMING (Raíces truncadas con SnowballStemmer):
   ['entr', 'compr', 'rsi', '28', '_nivel_sobrevent_', 'no', 'esper', 'cierr', 

---
## 6. Resumen de Experimentos Comparativos (Sección 5.2 y 5.3 del PDF)

### 6.1 Comparativa Stemming vs. Lematización y N-gramas
| Configuración del Pipeline NLP | Rango de N-gramas | Vocabulario Activo | Accuracy (Test) | F1-Score Macro (Test) | Decisión |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Stemming (`SnowballStemmer`)** | Unigramas `(1, 1)` | 412 términos | `0.9190` | `0.9215` | Descartado (trunca raíces y pierde contexto de negación) |
| **Lematización (`mode='lemma'`)** | Unigramas `(1, 1)` | 398 términos | `0.9286` | `0.9310` | Supera a Stemming en legibilidad y generalización |
| **Lematización + Bigramas (Elegido)** | **Unigramas + Bigramas `(1, 2)`** | **1,845 términos** | **`0.9619`** | **`0.9638`** | **ELEGIDO EN PRODUCCIÓN** (captura `"no esperar"`, `"cierre vela"`, `"sin stop"`) |

### 6.2 Comparativa de los 2 Modelos de Machine Learning (`MultinomialNB` vs `LogisticRegression`)
| Modelo Evaluado | Hiperparámetro Óptimo (`GridSearchCV` 5-Fold) | Accuracy | Precision Macro | Recall Macro | F1-Score Macro | Rol Final |
| :--- | :--- | :---: | :---: | :---: | :---: | :--- |
| **Modelo 1: `MultinomialNB` (Naive Bayes)** | `alpha = 0.5` | `0.9452` | `0.9481` | `0.9460` | `0.9469` | Baseline probabilístico comparativo |
| **Modelo 2: `LogisticRegression` (Producción)** | `C = 2.0, class_weight = 'balanced'` | **`0.9619`** | **`0.9645`** | **`0.9632`** | **`0.9638`** | **MODELO EN PRODUCCIÓN (`pipeline_prod.joblib`)** |

> **Conclusión Técnica:** `LogisticRegression` con Lematización y Bigramas `(1, 2)` supera a `MultinomialNB` porque no asume independencia condicional entre palabras: cuando una bitácora contiene términos positivos de estrategia (`"compra"`, `"rsi"`, `"sobreventa"`) junto con un error operativo (`"no esperar"`, `"cierre vela"`), la Regresión Logística asigna mayor peso discriminativo al bigrama de error, clasificando correctamente el riesgo.